## 测试 pydantic嵌套


In [1]:
from typing import Union

from langchain.agents.structured_output import ToolStrategy,MultipleStructuredOutputsError,StructuredOutputValidationError
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
# 1.初始化模型
model = ChatOpenAI(
    # model="carstenuhlig/omnicoder-9b:latest",
    # model="483025889/qwen3.5:9b",
    # model="mistral-nemo:latest",
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

## 定义错误处理函数
def custom_error_handler(error: Exception) ->str:
    """自定义错误处理函数"""
    err_str = str(error)
    print(f"捕获到错误类型{type(error).__name__}")
    print(f"错误详情{err_str}")
    if isinstance(error, StructuredOutputValidationError):
        return "评价数据格式有误，请检查字段是否符合要求."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回"
    else:
        return f"Error: {err_str}"

class ContactInfo(BaseModel):
    """联系人信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")

class EnventDetails(BaseModel):
    """活动详情"""
    contact: ContactInfo = Field(description="联系人信息") # 嵌套类是没有问题的。
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EnventDetails], ## 注意Union里面的东西只能取一个
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler
    )
)


result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"请提取以下文本中的内容：姓名:张三，电子邮箱：zhang3@atguigu.com,活动名称：公司年会，活动日期：2026-7-15"
    }]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中的内容：姓名:张三，电子邮箱：zhang3@atguigu.com,活动名称：公司年会，活动日期：
2026-7-15',
            additional_kwargs={},
            response_metadata={},
            id='7f777e4b-b3eb-449b-9c7d-da131c2059a6'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 391,
                    'prompt_tokens': 357,
                    'total_tokens': 748,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 152}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3-vl:latest',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-976',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a11e89-ef29-7411-b33a-60e9462acfef-0',
            tool_calls=[
                {
                    'name': 'EnventDetails',
                    'args': {
                        'contact': {'email': 'zhang3@atguigu.com', 'name': '张三'},
                        'event_name': '公司年会',
                        'date': '2026-7-15'
                    },
                    'id': 'call_mlvz062n',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 357,
                'output_tokens': 391,
                'total_tokens': 748,
                'input_token_details': {'cache_read': 152},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='提取完成！',
            name='EnventDetails',
            id='a7129417-d5a2-4680-ad15-6d471fd65698',
            tool_call_id='call_mlvz062n'
        )
    ],
    'structured_response': EnventDetails(
        contact=ContactInfo(name='张三', email='zhang3@atguigu.com'),
        event_name='公司年会',
        date='2026-7-15'
    )
}